# Arabisch → Deutsch mit Sprachausgabe

**Pipeline:** Arabische Audiodatei → **Cohere Transcribe Arabic** (ASR) → **TranslateGemma 12B** (Übersetzung) → **F5-TTS German** (deutsche Sprachausgabe als WAV).

**Ausführen:** `Laufzeit > Laufzeittyp ändern > T4 GPU`, dann `Laufzeit > Alle ausführen`.

1. **Hugging Face Token:** als Colab-Secret `HF_TOKEN` (Schlüssel-Symbol links) oder per `.env`-Upload. Vorher auf huggingface.co die Nutzungsbedingungen von `CohereLabs/cohere-transcribe-arabic-07-2026` und `google/translategemma-12b-it` akzeptieren.
2. **Arabische Audiodatei** (wav, mp3, m4a, ...) hochladen, wenn danach gefragt wird.
3. **Stimme (optional):** Standardmäßig spricht die mitgelieferte F5-Beispielstimme. Für eine eigene Stimme in Zelle 4 `EIGENE_STIMME` anhaken, eine 5–10 s lange deutsche WAV hochladen und den exakt gesprochenen Text in `REF_TEXT` eintragen.
4. Am Ende werden arabischer Text, deutsche Übersetzung und die deutsche Audiodatei angezeigt und zum Download angeboten.

Die Modelle werden nacheinander geladen und danach wieder aus dem GPU-Speicher entfernt, damit alles auf eine T4 (15 GB) passt.

In [ ]:
#@title 1. GPU prüfen und Pakete installieren
import subprocess, sys

import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "Keine GPU: Laufzeit > Laufzeittyp ändern > 'T4 GPU' > Speichern, dann 'Laufzeit > Alle ausführen'. "
        "Bleibt der Fehler, ist das kostenlose GPU-Kontingent für heute aufgebraucht."
    )
print("GPU:", torch.cuda.get_device_name(0))

PACKAGES = [
    "transformers>=5.4.0", "accelerate", "bitsandbytes>=0.45", "f5-tts", "huggingface_hub",
    "soundfile", "librosa", "sentencepiece", "protobuf", "python-dotenv",
]
IMPORT_CHECK = (
    "import f5_tts.api, soundfile, librosa, bitsandbytes, dotenv; "
    "from transformers import AutoProcessor, CohereAsrForConditionalGeneration, AutoModelForImageTextToText"
)

result = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input", *PACKAGES],
                        capture_output=True, text=True)
if result.returncode != 0:
    print(result.stdout[-3000:])
    print(result.stderr[-3000:])
    raise RuntimeError("Installation fehlgeschlagen (siehe oben).")

check = subprocess.run([sys.executable, "-c", IMPORT_CHECK], capture_output=True, text=True)
if check.returncode != 0:
    print(check.stderr[-3000:])
    raise RuntimeError("Pakete installiert, aber nicht importierbar (siehe oben).")
print("OK: Cohere ASR, TranslateGemma und F5-TTS sind installiert.")

In [ ]:
#@title 2. Hugging Face Token
import os
from pathlib import Path

from dotenv import dotenv_values
from huggingface_hub import hf_hub_download, login

ASR_MODEL = "CohereLabs/cohere-transcribe-arabic-07-2026"
MT_MODEL = "google/translategemma-12b-it"


def find_token():
    try:
        from google.colab import userdata
        token = (userdata.get("HF_TOKEN") or "").strip()
        if token:
            return token
    except Exception:
        pass
    for path in [Path("/content/.env"), *sorted(Path("/content").glob("*env*"))]:
        if path.is_file():
            token = (dotenv_values(path).get("HF_TOKEN") or "").strip()
            if token:
                return token
    from google.colab import files
    print("Kein HF_TOKEN gefunden. Bitte die .env mit HF_TOKEN=... hochladen:")
    for name in files.upload():
        token = (dotenv_values(Path("/content") / Path(name).name).get("HF_TOKEN") or "").strip()
        if token:
            return token
    return ""


HF_TOKEN = find_token()
if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN fehlt: als Colab-Secret anlegen oder eine .env mit HF_TOKEN=... hochladen.")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN, add_to_git_credential=False)

for repo in (ASR_MODEL, MT_MODEL):
    try:
        hf_hub_download(repo, "config.json", token=HF_TOKEN)
    except Exception as exc:
        raise RuntimeError(
            f"Kein Zugriff auf {repo}: auf huggingface.co/{repo} die Bedingungen akzeptieren "
            f"und prüfen, ob der Token Leserechte hat. ({exc})"
        ) from exc
print("OK: Token gültig, Zugriff auf Cohere ASR und TranslateGemma vorhanden.")

In [ ]:
#@title 3. Arabische Audiodatei hochladen
from pathlib import Path

import librosa
from google.colab import files
from IPython.display import Audio, display

SAMPLE_RATE = 16_000

print("Arabische Audiodatei auswählen (wav, mp3, m4a, ...):")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Keine Datei hochgeladen.")
INPUT_PATH = str(Path("/content") / Path(next(iter(uploaded))).name)

audio_ar, _ = librosa.load(INPUT_PATH, sr=SAMPLE_RATE, mono=True)
print(f"{Path(INPUT_PATH).name}: {len(audio_ar) / SAMPLE_RATE:.1f} s")
display(Audio(audio_ar, rate=SAMPLE_RATE))

In [ ]:
#@title 4. Stimme für die deutsche Sprachausgabe
#@markdown Ohne Haken wird die mitgelieferte F5-TTS-Beispielstimme verwendet.
#@markdown Mit Haken: 5–10 s deutsche WAV hochladen und den exakt gesprochenen Text eintragen
#@markdown (leer = F5-TTS transkribiert den Clip selbst mit Whisper).
EIGENE_STIMME = False  #@param {type:"boolean"}
REF_TEXT = ""  #@param {type:"string"}

from importlib.resources import files as package_files
from pathlib import Path

if EIGENE_STIMME:
    from google.colab import files
    print("Referenz-WAV der gewünschten Stimme auswählen:")
    uploaded_ref = files.upload()
    if not uploaded_ref:
        raise RuntimeError("Keine Referenzdatei hochgeladen.")
    REF_WAV = str(Path("/content") / Path(next(iter(uploaded_ref))).name)
    REF_TEXT = REF_TEXT.strip()
else:
    REF_WAV = str(package_files("f5_tts").joinpath("infer/examples/basic/basic_ref_en.wav"))
    REF_TEXT = "Some call me nature, others call me mother nature."

if not Path(REF_WAV).is_file():
    raise RuntimeError(f"Referenzdatei nicht gefunden: {REF_WAV}")
print("Referenzstimme:", REF_WAV)
print("Referenztext:", REF_TEXT or "(wird automatisch transkribiert)")

In [ ]:
#@title 5. Spracherkennung Arabisch (Cohere Transcribe)
import gc

import torch
from transformers import AutoProcessor, CohereAsrForConditionalGeneration

DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()


print("Lade", ASR_MODEL, "...")
asr_processor = AutoProcessor.from_pretrained(ASR_MODEL, token=HF_TOKEN)
asr_model = CohereAsrForConditionalGeneration.from_pretrained(
    ASR_MODEL, dtype=DTYPE, device_map="auto", token=HF_TOKEN
).eval()

# Audio longer than ~30 s is split by the processor and reassembled via audio_chunk_index.
inputs = asr_processor(audio_ar, sampling_rate=SAMPLE_RATE, return_tensors="pt", language="ar")
audio_chunk_index = inputs.get("audio_chunk_index")
inputs = inputs.to(asr_model.device, dtype=asr_model.dtype)
with torch.inference_mode():
    outputs = asr_model.generate(**inputs, max_new_tokens=256)
decoded = asr_processor.decode(
    outputs, skip_special_tokens=True, audio_chunk_index=audio_chunk_index, language="ar"
)
text_ar = (" ".join(decoded) if isinstance(decoded, list) else decoded).strip()

asr_model = asr_processor = inputs = outputs = None
free_gpu()

if not text_ar:
    raise RuntimeError("Keine Sprache erkannt. Ist die Datei arabisch und hörbar?")
Path("/content/transkript_ar.txt").write_text(text_ar, encoding="utf-8")
print("\nArabisch:\n" + text_ar)

In [ ]:
#@title 6. Übersetzung Arabisch → Deutsch (TranslateGemma 12B, 4-bit)
import re

from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

MAX_CHARS_PER_REQUEST = 600


def split_sentences(text, max_chars):
    sentences = [s.strip() for s in re.split(r"(?<=[.!?؟])\s+|\n+", text) if s.strip()]
    chunks, current = [], ""
    for sentence in sentences:
        if current and len(current) + len(sentence) + 1 > max_chars:
            chunks.append(current)
            current = sentence
        else:
            current = f"{current} {sentence}".strip()
    if current:
        chunks.append(current)
    return chunks


def translate(text):
    messages = [{
        "role": "user",
        "content": [{"type": "text", "source_lang_code": "ar", "target_lang_code": "de", "text": text}],
    }]
    batch = mt_processor.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt"
    ).to(mt_model.device)
    input_len = batch["input_ids"].shape[1]
    with torch.inference_mode():
        output = mt_model.generate(**batch, max_new_tokens=512, do_sample=False)
    return mt_processor.decode(output[0, input_len:], skip_special_tokens=True).strip()


print("Lade", MT_MODEL, "(4-bit) ...")
mt_processor = AutoProcessor.from_pretrained(MT_MODEL, token=HF_TOKEN)
mt_model = AutoModelForImageTextToText.from_pretrained(
    MT_MODEL,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=DTYPE,
    ),
    device_map="auto",
    token=HF_TOKEN,
).eval()

text_de = " ".join(translate(chunk) for chunk in split_sentences(text_ar, MAX_CHARS_PER_REQUEST)).strip()

mt_model = mt_processor = None
free_gpu()

if not text_de:
    raise RuntimeError("Die Übersetzung ist leer.")
Path("/content/uebersetzung_de.txt").write_text(text_de, encoding="utf-8")
print("\nDeutsch:\n" + text_de)

In [ ]:
#@title 7. Deutsche Sprachausgabe (F5-TTS German) und Download
import numpy as np
import soundfile as sf
from f5_tts.api import F5TTS
from google.colab import files
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

F5_GERMAN_REPO = "aihpi/F5-TTS-German"
F5_GERMAN_CKPT = "F5TTS_Base/model_420000.safetensors"
OUTPUT_WAV = "/content/uebersetzung_de.wav"

print("Lade F5-TTS German ...")
tts = F5TTS(
    model="F5TTS_Base",
    ckpt_file=hf_hub_download(F5_GERMAN_REPO, F5_GERMAN_CKPT),
    vocab_file=hf_hub_download(F5_GERMAN_REPO, "vocab.txt"),
    device="cuda:0",
)

wav, tts_rate, _ = tts.infer(
    ref_file=REF_WAV,
    ref_text=REF_TEXT,
    gen_text=text_de,
    nfe_step=32,
    remove_silence=False,
)
wav = np.asarray(wav, dtype=np.float32).reshape(-1)
sf.write(OUTPUT_WAV, wav, tts_rate)

print("\nArabisch:\n" + text_ar)
print("\nDeutsch:\n" + text_de)
print(f"\nDeutsche Audiodatei: {OUTPUT_WAV} ({len(wav) / tts_rate:.1f} s)")
display(Audio(wav, rate=tts_rate))

for path in (OUTPUT_WAV, "/content/uebersetzung_de.txt", "/content/transkript_ar.txt"):
    files.download(path)